# Task classification of saved object summaries

Read the `_semantics.json` results produced by
[the object-summary batch notebook](05.application.summary.object.ipynb), then
classify task tables using
[core.dom.summary.task](01.core.dom.summary.task.ipynb).

Each complete result is saved alongside its input as `<markdown_stem>_tasks.json`.
The document structure and every summary are preserved; qualifying table object
types become `task`. Original semantics, Markdown, AST caches, partial progress
and object-summary checkpoints remain untouched. No model client, provider
preflight or summary generation is involved.

## Configuration

Use the same two corpus directories as the object-summary notebook, in 4.0 then
5.0 order. `PROJ_ROOT` may be set in the environment; otherwise find the project
from the current working directory. A `.env` file and model credentials are not
required. Export the new core notebook with nbdev before using this notebook.

`PREVIEW_OUTPUTS` and `PREVIEW_TASKS` bound only the saved-output preview, never
corpus processing. Creating and validating this notebook does not run this batch.

In [ ]:
# | notest
import os
import sys
from pathlib import Path


def resolve_project_root():
    configured = os.getenv('PROJ_ROOT')
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / 'pyproject.toml').is_file():
            raise FileNotFoundError(f'Invalid PROJ_ROOT (missing pyproject.toml): {root}')
        return root
    cwd = Path.cwd().resolve()
    for root in (cwd, *cwd.parents):
        if (root / 'pyproject.toml').is_file():
            return root
    raise FileNotFoundError('Cannot locate PROJ_ROOT containing pyproject.toml')


PROJ_ROOT = resolve_project_root()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

INPUT_DIRS = [
    PROJ_ROOT / 'assets' / 'code-docs.4.0' / '.md',
    PROJ_ROOT / 'assets' / 'code-docs.5.0' / '.md',
]
PREVIEW_OUTPUTS = 3
PREVIEW_TASKS = 20
print('Saved-summary inputs:')
for folder in INPUT_DIRS:
    print(f'  {folder}')
print('Output: adjacent <markdown_stem>_tasks.json files')

## Offline batch helpers

Discovery validates every input directory before any output is written and
excludes `_tasks.json` and `_summary-progress.json` files. Repeated or overlapping
directories do not process a document twice.

Each file is classified fully in memory, then written to a temporary file in the
same directory and published with atomic replacement. Equal existing results are
not rewritten. An existing malformed task output can be repaired by a successful
new classification. Errors preserve the previous output and are reported per
file while the batch continues. Interrupting stops processing and cleans up
unpublished temporary output; already published files remain available for reruns.

In [ ]:
from pathlib import Path
from time import perf_counter
import json
import os
import tempfile

from tqdm.auto import tqdm
from ribosome.core.dom.summary.task import classify_task_objects, task_object_records


def discover_semantics(input_dirs):
    """Validate all corpora, then return sorted, deduplicated saved summaries."""
    if isinstance(input_dirs, (str, Path)):
        input_dirs = [input_dirs]
    input_dirs = list(input_dirs)
    if not input_dirs:
        raise ValueError('INPUT_DIRS must contain at least one folder')
    corpora, seen = [], set()
    for value in input_dirs:
        folder = Path(value).expanduser().resolve()
        if not folder.is_dir():
            raise FileNotFoundError(f'Semantics input directory does not exist: {folder}')
        files = sorted({path.resolve() for path in folder.rglob('*_semantics.json') if path.is_file()})
        if not files:
            raise ValueError(f'No *_semantics.json files found under: {folder}')
        unique = [file for file in files if file not in seen]
        seen.update(unique)
        corpora.append((folder, unique))
    return corpora


def task_output_path(semantics_file):
    semantics_file = Path(semantics_file)
    suffix = '_semantics.json'
    if not semantics_file.name.endswith(suffix):
        raise ValueError(f'Expected a {suffix} input: {semantics_file}')
    return semantics_file.with_name(semantics_file.name[:-len(suffix)] + '_tasks.json')


def write_task_output(output, tree):
    """Atomically publish one full tree; return False for an unchanged output."""
    output = Path(output)
    if output.is_file():
        try:
            if json.loads(output.read_text(encoding='utf-8')) == tree:
                return False
        except (json.JSONDecodeError, UnicodeError):
            pass  # A complete new result can repair a malformed task output.
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', dir=output.parent,
                                         prefix=f'.{output.name}.', suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            json.dump(tree, stream, ensure_ascii=False, indent=2)
            stream.write('\n')
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, output)
        return True
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def classify_task_files(corpora, *, results=None, progress=True):
    """Classify sequentially, retaining finished reports if interrupted.

    File errors are reported and other documents continue. KeyboardInterrupt
    propagates after the atomic writer cleans up its unpublished temporary file.
    """
    if results is None:
        results = []
    counts = dict(written=0, unchanged=0, failed=0)
    with tqdm(total=sum(len(files) for _, files in corpora), desc='Task classification',
              unit='file', disable=not progress) as bar:
        for folder, files in corpora:
            for file in files:
                started = perf_counter()
                row = dict(corpus=str(folder), input=str(file), output='',
                           status='failed', converted_tables=0, task_count=0, error='')
                bar.set_postfix(file=Path(file).name, **counts)
                try:
                    output = task_output_path(file)
                    row['output'] = str(output)
                    tree = json.loads(Path(file).read_text(encoding='utf-8'))
                    if not isinstance(tree, dict) or not isinstance(tree.get('blocks'), list):
                        raise ValueError('Expected an object-summary document with a blocks list')
                    records = task_object_records(tree)
                    classified = classify_task_objects(tree)
                    written = write_task_output(output, classified)
                    row.update(status='written' if written else 'unchanged',
                               converted_tables=sum(record['type'] == 'table' for record in records),
                               task_count=len(records))
                except Exception as exc:
                    row['error'] = f'{type(exc).__name__}: {exc}'
                    bar.write(f'FAILED: {file}\n{row["error"]}')
                row['elapsed_seconds'] = perf_counter() - started
                results.append(row)
                counts[row['status']] += 1
                bar.set_postfix(file=Path(file).name, **counts)
                bar.update(1)
    return results

## Discover saved summaries

This cell reads directory listings only. Missing or empty corpora stop discovery
before processing begins. All saved summary documents are included, even when
their original Markdown is unavailable.

In [ ]:
# | notest
corpora = discover_semantics(INPUT_DIRS)
for folder, files in corpora:
    print(f'{folder}: {len(files)} saved summaries')
print(f'Total: {sum(len(files) for _, files in corpora)}')

## Classify and publish task outputs

Run this cell to process all discovered files. `converted_tables` counts unique
tables classified in the current input, including on an unchanged rerun;
`task_count` also includes preexisting tasks. `written`, `unchanged` and `failed`
describe output publication. The external `results` list retains completed-file
reports if the cell is interrupted. No separate task checkpoint is needed:
reruns re-evaluate saved inputs and skip equal outputs.

In [ ]:
# | notest
results = []
classify_task_files(corpora, results=results)

## Results and bounded preview

Review failures before rerunning. The preview reads successful task outputs and
shows unique tasks with source lines, section paths and matching reasons. Existing
tasks that no longer match a rule remain tasks and have no current matching reason.
Diagnostics are displayed here and are not added to the saved document.

In [ ]:
# | notest
import pandas as pd
from IPython.display import display

results_df = pd.DataFrame(results)
display(results_df)
if not results_df.empty:
    print(results_df.groupby(['corpus', 'status']).size().to_string())
    print(f"Tables classified: {results_df['converted_tables'].sum()}")
    failures = results_df.loc[results_df['status'] == 'failed']
    if not failures.empty:
        display(failures[['input', 'error']])

successful = [row for row in results if row['status'] in {'written', 'unchanged'}]
for row in successful[:max(0, PREVIEW_OUTPUTS)]:
    print(row['output'])
    saved = json.loads(Path(row['output']).read_text(encoding='utf-8'))
    records = task_object_records(saved)
    preview = pd.DataFrame(records[:max(0, PREVIEW_TASKS)])
    if not preview.empty:
        preview['section_path'] = preview['section_path'].map(lambda path: ' / '.join(path))
        preview['reasons'] = preview['reasons'].map(lambda reasons: ', '.join(reasons))
    print(f'{len(records)} tasks; showing {len(preview)}')
    display(preview)